# Advanced OCR — Colab runner (100% OXR-1.0)

This notebook is **only a runner**. All code lives in the GitHub repo
`greencube-dev-boop/advanced-ocr` (synced below with `git pull` — push updates to
GitHub and just rerun this notebook, never re-upload it). All data lives in your
Google Drive under **`OCR Engine/`**.

Pipeline: raw PDF → **every page is parsed by SoMarkAI/OXR-1.0** (tables → Markdown,
formulas → LaTeX, chemical equations → mhchem, chemical structures → SMILES,
code blocks → fences — the native PDF text layer is not used at all) → one
Markdown per book in `OCR Engine/refined/`.

**No OCR/API keys are needed.** The default `oxr.mode=free_api` uses the free OXR API
(no sign-up; ≤10MB/≤20 pages per request — the pipeline chunks accordingly).
Self-hosting OXR (`oxr.mode=local`) needs a ≥24GB BF16 GPU — see the notes cell below.
The only token you may ever need is a **GitHub PAT (read access)** to clone this
**private** repo — put it in Colab Secrets as `GITHUB_TOKEN` (🔑 sidebar, recommended)
or in the sync cell below.

**Rerunning this notebook resumes where it stopped** (state: `OCR Engine/config/books.json`).
To re-convert a book 100% with OXR after upgrading, run `ocr_pipeline reset <book>` first.

In [ ]:
# 1) Dependencies (idempotent)
!pip install -q PyMuPDF requests tqdm
print("deps ok")

In [ ]:
# 2) Mount Google Drive and set the OCR Engine root
import os
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/OCR Engine"
os.makedirs(DRIVE_ROOT, exist_ok=True)   # creates raw/ config/ refined/ temp/ on first use
print("Drive root:", DRIVE_ROOT)
for sub in ("raw", "config", "refined", "temp"):
    os.makedirs(os.path.join(DRIVE_ROOT, sub), exist_ok=True)
print("raw/ is where you drop PDFs:", os.path.join(DRIVE_ROOT, "raw"))

In [ ]:
# 3) Sync the codebase from GitHub (clone once, then git pull on every run)
#    The repo is PRIVATE: a GitHub PAT with read access is required.
#    Recommended: Colab Secrets (key icon in the left sidebar) -> add GITHUB_TOKEN.
#    Fallback: paste the PAT into GITHUB_TOKEN below.
#    Never save this notebook back to GitHub with the token filled in.
import os, shutil, subprocess, sys

REPO_URL = "https://github.com/greencube-dev-boop/advanced-ocr.git"
GITHUB_TOKEN = ""
WORKDIR = "/content/advanced-ocr"

try:
    from google.colab import userdata
    GITHUB_TOKEN = GITHUB_TOKEN or userdata.get("GITHUB_TOKEN")
    print("token source: Colab Secrets")
except Exception:
    print("token source:", "in-cell variable" if GITHUB_TOKEN else "NONE (anonymous clone)")

url = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL

if os.path.isdir(os.path.join(WORKDIR, ".git")):
    subprocess.run(["git", "-C", WORKDIR, "pull", "--ff-only", url], check=False)
else:
    shutil.rmtree(WORKDIR, ignore_errors=True)   # clear any partial clone
    r = subprocess.run(["git", "clone", url, WORKDIR])
    if r.returncode != 0:
        shutil.rmtree(WORKDIR, ignore_errors=True)
        raise SystemExit(
            f"git clone failed (exit {r.returncode}). The repo is PRIVATE — do ONE of:
"
            "  a) Colab Secrets: add GITHUB_TOKEN = your GitHub PAT (recommended), then rerun this cell
"
            "  b) paste your PAT into GITHUB_TOKEN = "..." in this cell, then rerun
"
            "  c) make the repo public (the code contains no keys)")

os.environ["PYTHONPATH"] = os.path.join(WORKDIR, "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
sys.path.insert(0, os.path.join(WORKDIR, "src"))
import ocr_pipeline
print("pipeline", ocr_pipeline.__version__, "from", WORKDIR)

## About API keys and OXR modes

- **No OCR keys are required.** The free OXR API (`oxr.mode=free_api`, default) needs no sign-up.
  Limits per request: 10 MB / 20 pages — the pipeline builds chunks that respect both
  (oversized single pages are sent as rendered PNGs). The free API is queued and best-effort.
- **Self-hosting OXR** (`oxr.mode=local` in `Drive/OCR Engine/config/pipeline.json`): run the
  [OXR services](https://github.com/SoMarkAI/OXR) yourself (Docker all-in-one or the three-service
  source setup) and point `oxr.local_base_url` at it (default `http://127.0.0.1:8000/v1`).
  Requirements: Linux, NVIDIA GPU **≥24 GB with BF16 support** (free Colab T4 = 16 GB — NOT
  sufficient; Colab Pro A100/L4-class runtimes work), CUDA 13.0 driver, Python 3.12.

In [ ]:
# 4) Sanity check: deps, drive, OXR reachability
!python -m ocr_pipeline doctor --drive-root "{DRIVE_ROOT}"
!python -m ocr_pipeline init-config --drive-root "{DRIVE_ROOT}"

In [ ]:
# 5) Run / resume the pipeline.
#    Drop PDFs into Drive -> "OCR Engine/raw/" first.
#    Finished pages are skipped automatically; interrupt anytime and rerun to resume.
!python -m ocr_pipeline run --drive-root "{DRIVE_ROOT}"

In [ ]:
# 6) Inspect resume state
!python -m ocr_pipeline status --drive-root "{DRIVE_ROOT}"

## Outputs

- Combined book Markdown: `Drive/OCR Engine/refined/<book>.md`
- Per-page trail (final / meta / chunk JSON+MD): `Drive/OCR Engine/temp/<book>/`
- Resume state: `Drive/OCR Engine/config/books.json` — settings: `Drive/OCR Engine/config/pipeline.json`

## Tips

- Settings (OXR mode/URL, chunking, figure markers) are edited in
  `Drive/OCR Engine/config/pipeline.json` — no code changes needed.
- Reprocess a book from scratch (e.g. after an engine/config change):
  `!python -m ocr_pipeline reset mybook.pdf --drive-root "{DRIVE_ROOT}"` then run again.
- Process a single book: `!python -m ocr_pipeline run-book mybook.pdf --drive-root "{DRIVE_ROOT}"`.
